# Aplicações de Satélites, Espaços de Cores e Demonstração em Python

Este notebook demonstra o uso de imagens de satélite em diferentes espaços de cores,
com foco em **RGB** e **HSV**, utilizando um arquivo multibanda (por exemplo, Sentinel-2).

## 1. Aplicações de Satélite e Sensoriamento Remoto

Imagens de satélite são amplamente utilizadas em diversas áreas, como:

- **Agricultura** – monitoramento de safras, detecção de estresse hídrico.
- **Meio ambiente** – mapeamento de desmatamento, queimadas, mudanças na vegetação.
- **Gestão urbana** – análise de expansão urbana e uso do solo.
- **Desastres naturais** – acompanhamento de enchentes, deslizamentos e áreas críticas.

Essas aplicações dependem fortemente de como as **bandas espectrais** são combinadas e representadas em **espaços de cor**.

## 2. Espaços de Cores Comumente Utilizados em Sensoriamento Remoto

### 2.1. Espaço de Cor RGB
O espaço RGB (Red, Green, Blue) é o mais usado para exibição de imagens em tela.
No sensoriamento remoto, é utilizado para:

- **Composição de cor verdadeira** (true color)
- **Composição de falsa cor** (false color) destacando vegetação, água etc.

A seguir, um exemplo de **composição RGB (cor verdadeira)** a partir de uma imagem multibanda.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

# Caminho para a imagem multibanda (ex.: Sentinel-2)
# Substitua pelo caminho correto do seu arquivo .tif
caminho_imagem = 'sentinel2_multiband.tif'

with rasterio.open(caminho_imagem) as src:
    # Exemplo Sentinel-2: banda 4 = R, 3 = G, 2 = B
    red = src.read(4).astype(np.float32)
    green = src.read(3).astype(np.float32)
    blue = src.read(2).astype(np.float32)

# Empilhar as bandas em um array RGB (H x W x 3)
rgb = np.dstack([red, green, blue])

# Normalizar para o intervalo 0–1 para exibição
rgb_norm = (rgb - rgb.min()) / (rgb.max() - rgb.min())

plt.figure(figsize=(8, 8))
plt.title('Composição RGB (True Color)')
plt.imshow(rgb_norm)
plt.axis('off')
plt.show()

### 2.2. Espaço de Cor HSV
O espaço HSV (Hue, Saturation, Value) representa a cor de forma mais intuitiva:

- **Hue (matiz)** — tipo da cor
- **Saturation (saturação)** — pureza da cor
- **Value (valor)** — brilho

Ele é útil para **segmentação**, pois separa cor e intensidade. A seguir, a conversão da composição RGB para HSV e a visualização do canal de matiz.

In [ ]:
from skimage.color import rgb2hsv
import matplotlib.pyplot as plt

# Conversão de RGB normalizado para HSV
hsv = rgb2hsv(rgb_norm)

# Canal Hue (matiz)
hue = hsv[:, :, 0]

plt.figure(figsize=(8, 8))
plt.title('Canal Hue (Matiz) – Espaço HSV')
plt.imshow(hue, cmap='hsv')
plt.axis('off')
plt.show()

## 3. Composição Falsa Cor (NIR, R, G) – Ainda em RGB

Nesta composição, o infravermelho próximo (NIR) é colocado no canal vermelho,
o vermelho no canal verde e o verde no canal azul. Vegetação saudável costuma
aparecer em tons de vermelho forte, facilitando o monitoramento.


In [ ]:
with rasterio.open(caminho_imagem) as src:
    # Exemplo Sentinel-2: banda 8 = NIR
    nir = src.read(8).astype(np.float32)
    red = src.read(4).astype(np.float32)
    green = src.read(3).astype(np.float32)

# Composição falsa cor: NIR -> R, Red -> G, Green -> B
false_rgb = np.dstack([nir, red, green])
false_norm = (false_rgb - false_rgb.min()) / (false_rgb.max() - false_rgb.min())

plt.figure(figsize=(8, 8))
plt.title('Composição Falsa Cor (NIR, R, G)')
plt.imshow(false_norm)
plt.axis('off')
plt.show()

## 4. Segmentação Simples Usando o Canal Hue (HSV)

Com base no canal **Hue**, é possível criar uma máscara simples para destacar vegetação
ou outros alvos, utilizando intervalos de matiz aproximados.
Os valores de limiar abaixo são ilustrativos e podem precisar de ajuste conforme a cena.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Exemplo de máscara simples de vegetação com base no Hue
# Ajuste dos limites conforme a cena e a paleta resultante
vegetacao_mask = (hue > 0.2) & (hue < 0.5)

plt.figure(figsize=(8, 8))
plt.title('Máscara de Vegetação a partir do Hue')
plt.imshow(vegetacao_mask, cmap='gray')
plt.axis('off')
plt.show()